# Tạo dữ liệu Quick Link từ file .md",",

Notebook này quét nội dung các file `.md",",` (dựa vào tên file + heading) để tự sinh
phần `items` (title / slug / children.anchor) cho MỘT (nikaya, edition).

Những gì notebook **không** đoán được — vì đó là thuộc tính của cả bộ kinh/bản dịch,
không nằm trong nội dung 1 file — sẽ để `"???"` cho bạn tự điền:
`folder`, `label`, `path`, `index_length`.

**Quy ước mà notebook giả định** (chỉnh lại ở phần CONFIG nếu khác):
- Tên file (bỏ `.md",",`) là `slug`, dạng `<nikaya>-<số>-...`, ví dụ `mn-007-the-simile-of-the-cloth`
  → số thứ tự đầu tiên (`top index`) = `7`.
- Dòng `# ...` (H1) đầu tiên trong file là `title` của kinh.
- Các heading cấp `###` (H3, xem `CHILD_HEADING_LEVEL`) có dạng số đầu dòng như
  `### 2.2 Dutiyakassapasutta` được coi là mốc "đoạn con" → sinh `children`.
  Số đầu tiên trong heading con phải khớp với top index của file (nếu không sẽ có warning),
  các số còn lại tạo thành đường dẫn lồng nhau trong `children`.

**Về anchor:** notebook tự đoán anchor bằng một hàm slugify phổ biến (lowercase, thay
ký tự không phải chữ/số bằng `-`), khớp đúng với ví dụ `2-2-dutiyakassapasutta` bạn đưa.
Tuy nhiên đây là suy đoán — VitePress dùng `markdown-it-anchor` nội bộ và có thể xử lý
dấu Pali/Việt (ā, ṁ, ñ, ...) khác đi. **Nên đối chiếu lại với anchor thật** (mở trang đã
build, bấm vào link cạnh heading, xem `#...` trên thanh địa chỉ) trước khi dùng chính thức,
đặc biệt với các tiêu đề có dấu.

## 1. CONFIG — sửa các giá trị dưới đây

In [30]:
import os, re, json
# để làm pali-vi, đầu tiên lấy file của pali-vi, ra kết quả (json),
# maping giữa pali-vi và tmc và json của c-pali-tmc-vi (nhớ đổi mn-> mnc)


# Thư mục chứa các file .md",",
SOURCE_DIR = "../../docs/kinhtrungbo/thichminhchau"

# Danh sách tên file .md",", cần xử lý (chỉ tên file, không cần đường dẫn đầy đủ)
FILES = [
"mn-001-kinh-phap-mon-can-ban.md",
"mn-002-kinh-tat-ca-cac-lau-hoac.md",
"mn-003-kinh-thua-tu-phap.md",
"mn-004-kinh-so-hai-khiep-dam.md",
"mn-005-kinh-khong-ue-nhiem.md",
"mn-006-kinh-uoc-nguyen.md",
"mn-007-kinh-vi-du-tam-vai.md",
"mn-008-kinh-doan-giam.md",
"mn-009-kinh-chanh-tri-kien.md",
"mn-010-kinh-niem-xu.md",
"mn-011-tieu-kinh-su-tu-hong.md",
"mn-012-dai-kinh-su-tu-hong.md",
"mn-013-dai-kinh-kho-uan.md",
"mn-014-tieu-kinh-kho-uan.md",
"mn-015-kinh-tu-luong.md",
"mn-016-kinh-tam-hoang-vu.md",
"mn-017-kinh-khu-rung.md",
"mn-018-kinh-mat-hoan.md",
"mn-019-kinh-song-tam.md",
"mn-020-kinh-an-tru-tam.md",
"mn-021-kinh-vi-du-cai-cua.md",
"mn-022-kinh-vi-du-con-ran.md",
"mn-023-kinh-go-moi.md",
"mn-024-kinh-tram-xe.md",
"mn-025-kinh-bay-moi.md",
"mn-026-kinh-thanh-cau.md",
"mn-027-tieu-kinh-du-dau-chan-voi.md",
"mn-028-dai-kinh-du-dau-chan-voi.md",
"mn-029-dai-kinh-thi-du-loi-cay.md",
"mn-030-tieu-kinh-du-loi-cay.md",
"mn-031-tieu-kinh-rung-sung-bo.md",
"mn-032-dai-kinh-rung-sung-bo.md",
"mn-033-dai-kinh-nguoi-chan-bo.md",
"mn-034-tieu-kinh-nguoi-chan-bo.md",
"mn-035-tieu-kinh-saccaka.md",
"mn-036-dai-kinh-saccaka.md",
"mn-037-tieu-kinh-doan-tan-ai.md",
"mn-038-dai-kinh-doan-tan-ai.md",
"mn-039-dai-kinh-xom-ngua.md",
"mn-040-tieu-kinh-xom-ngua.md",
"mn-041-kinh-saleyyaka.md",
"mn-042-kinh-veranjaka.md",
"mn-043-dai-kinh-phuong-quang.md",
"mn-044-tieu-kinh-phuong-quang.md",
"mn-045-tieu-kinh-phap-hanh.md",
"mn-046-dai-kinh-phap-hanh.md",
"mn-047-kinh-tu-sat.md",
"mn-048-kinh-kosambiya.md",
"mn-049-kinh-pham-thien-cau-thinh.md",
"mn-050-kinh-hang-ma.md",
"mn-051-kinh-kandaraka.md",
"mn-052-kinh-bat-thanh.md",
"mn-053-kinh-huu-hoc.md",
"mn-054-kinh-potaliya.md",
"mn-055-kinh-jivaka.md",
"mn-056-kinh-uu-ba-ly.md",
"mn-057-kinh-hanh-con-cho.md",
"mn-058-kinh-vuong-tu-vo-uy.md",
"mn-059-kinh-nhieu-cam-tho.md",
"mn-060-kinh-khong-gi-chuyen-huong.md",
"mn-061-kinh-giao-gioi-la-hau-la-o-rung-ambala.md",
"mn-062-dai-kinh-giao-gioi-la-hau-la.md",
"mn-063-tieu-kinh-malunkya.md",
"mn-064-dai-kinh-malunkya.md",
"mn-065-kinh-bhaddali.md",
"mn-066-kinh-vi-du-con-chim-cay.md",
"mn-067-kinh-catuma.md",
"mn-068-kinh-nalakapana.md",
"mn-069-kinh-gulisani.md",
"mn-070-kinh-kitagiri.md",
"mn-071-kinh-day-vacchagotta-ve-tam-minh.md",
"mn-072-kinh-day-vacchagotta-ve-lua.md",
"mn-073-dai-kinh-vacchaghotta.md",
"mn-074-kinh-truong-trao.md",
"mn-075-kinh-magandiya.md",
"mn-076-kinh-sandaka.md",
"mn-077-dai-kinh-sakuludayi.md",
"mn-078-kinh-samanamandika.md",
"mn-079-tieu-kinh-sakuludayi-thien-sanh-uu-da-di.md",
"mn-080-kinh-vekhanassa.md",
"mn-081-kinh-ghatikara.md",
"mn-082-kinh-ratthapala.md",
"mn-083-kinh-makhadeva.md",
"mn-084-kinh-madhura.md",
"mn-085-kinh-vuong-tu-bo-de.md",
"mn-086-kinh-angulimala.md",
"mn-087-kinh-ai-sanh.md",
"mn-088-kinh-bahitika.md",
"mn-089-kinh-phap-trang-nghiem.md",
"mn-090-kinh-kannakatthala.md",
"mn-091-kinh-brahmayu.md",
"mn-092-kinh-sela.md",
"mn-093-kinh-assalayana.md",
"mn-094-kinh-ghotamukha.md",
"mn-095-kinh-canki.md",
"mn-096-kinh-esukari.md",
"mn-097-kinh-dhananjani.md",
"mn-098-kinh-vasettha.md",
"mn-099-kinh-subha.md",
"mn-100-kinh-sangarava.md",
"mn-101-kinh-devadaha.md",
"mn-102-kinh-nam-ba.md",
"mn-103-kinh-nghi-nhu-the-nao.md",
"mn-104-kinh-lang-sama.md",
"mn-105-kinh-thien-tinh.md",
"mn-106-kinh-bat-dong-loi-ich.md",
"mn-107-kinh-ganaka-moggallana.md",
"mn-108-kinh-gopaka-moggallana.md",
"mn-109-dai-kinh-man-nguyet.md",
"mn-110-tieu-kinh-man-nguyet.md",
"mn-111-kinh-bat-doan.md",
"mn-112-kinh-sau-thanh-tinh.md",
"mn-113-kinh-chan-nhan.md",
"mn-114-kinh-nen-hanh-tri-khong-nen-hanh-tri.md",
"mn-115-kinh-da-gioi.md",
"mn-116-kinh-thon-tien.md",
"mn-117-dai-kinh-bon-muoi.md",
"mn-118-kinh-nhap-tuc-xuat-tuc-niem.md",
"mn-119-kinh-than-hanh-niem.md",
"mn-120-kinh-hanh-sanh.md",
"mn-121-kinh-tieu-khong.md",
"mn-122-kinh-dai-thong.md",
"mn-123-kinh-hy-huu-vi-tang-huu-phap.md",
"mn-124-kinh-bac-cau-la.md",
"mn-125-kinh-dieu-ngu-dia.md",
"mn-126-kinh-phu-di.md",
"mn-127-kinh-a-na-luat.md",
"mn-128-kinh-tuy-phien-nao.md",
"mn-129-kinh-hien-ngu.md",
"mn-130-kinh-thien-xu.md",
"mn-131-kinh-nhat-da-hien-gia.md",
"mn-132-kinh-a-nan-nhat-da-hien-gia.md",
"mn-133-kinh-dai-ca-chien-dien-nhat-da-hien-gia.md",
"mn-134-kinh-lomasakangiya-nhat-da-hien-gia.md",
"mn-135-tieu-kinh-nghiep-phan-biet.md",
"mn-136-dai-kinh-nghiep-phan-biet.md",
"mn-137-kinh-phan-biet-sau-xu.md",
"mn-138-kinh-tong-thuyet-biet-thuyet.md",
"mn-139-kinh-vo-tranh-phan-biet.md",
"mn-140-kinh-gioi-phan-biet.md",
"mn-141-kinh-phan-biet-ve-su-that.md",
"mn-142-kinh-phan-biet-cung-duong.md",
"mn-143-kinh-giao-gioi-cap-co-doc.md",
"mn-144-kinh-giao-gioi-channa.md",
"mn-145-kinh-giao-gioi-phu-lau-na.md",
"mn-146-kinh-giao-gioi-nandaka.md",
"mn-147-tieu-kinh-giao-gioi-la-hau-la.md",
"mn-148-kinh-sau-sau.md",
"mn-149-dai-kinh-sau-xu.md",
"mn-150-kinh-noi-cho-dan-nagaravinda.md",
"mn-151-kinh-khat-thuc-thanh-tinh.md",
"mn-152-kinh-can-tu-tap.md",
]

# Heading cấp mấy được coi là "đoạn con" (### = 3, ## = 2, ...)
CHILD_HEADING_LEVEL = 3


## 2. Các hàm xử lý

In [31]:
TOP_INDEX_RE = re.compile(r'^[a-z]+-0*(\d+)')
H1_RE = re.compile(r'^#\s+(.*)$', re.MULTILINE)
CHILD_RE = re.compile(r'^#{%d}\s+(.*)$' % CHILD_HEADING_LEVEL)
NUM_PREFIX_RE = re.compile(r'^(\d+(?:\.\d+)+)\.?\s*')


def slugify(text):
    """Đoán anchor kiểu markdown-it-anchor. Xem ghi chú ở đầu notebook."""
    text = text.strip()
    text = re.sub(r'`([^`]*)`', r'\1', text)      # bỏ backtick code
    text = text.lower()
    text = re.sub(r'[^\w\s-]', '-', text, flags=re.UNICODE)
    text = re.sub(r'[\s_]+', '-', text)
    text = re.sub(r'-+', '-', text)
    return text.strip('-')


def top_index_from_slug(slug):
    m = TOP_INDEX_RE.match(slug.lower())
    return m.group(1) if m else None


def extract_h1(text):
    m = H1_RE.search(text)
    return m.group(1).strip() if m else None


def insert_child(item, path, anchor):
    node = item
    for i, key in enumerate(path):
        node.setdefault("children", {})
        node["children"].setdefault(key, {})
        node = node["children"][key]
        if i == len(path) - 1:
            node["anchor"] = anchor


def process_file(dirpath, filename, items, warnings):
    slug = filename[:-3] if filename.endswith(".md") else filename
    filepath = os.path.join(dirpath, filename)
    if not os.path.isfile(filepath):
        warnings.append(f"{filename}: không tìm thấy file, bỏ qua")
        return

    with open(filepath, "r", encoding="utf-8") as f:
        text = f.read()

    top_index = top_index_from_slug(slug)
    title = extract_h1(text)

    item = {}
    item["title"] = title if title else "??? (không tìm thấy H1)"
    if not title:
        warnings.append(f"{filename}: không tìm thấy dòng H1 (# ...)")
    item["slug"] = slug

    for line in text.splitlines():
        m = CHILD_RE.match(line)
        if not m:
            continue
        heading_text = m.group(1).strip()
        num_match = NUM_PREFIX_RE.match(heading_text)
        if not num_match:
            continue  # heading cấp con nhưng không có số đầu dòng -> bỏ qua
        numbers = num_match.group(1).split(".")
        if top_index and numbers[0] != str(int(top_index)):
            warnings.append(
                f'{filename}: heading "{heading_text}" có số đầu ({numbers[0]}) '
                f'khác top index suy từ tên file ({top_index}) — kiểm tra lại'
            )
        remaining = numbers[1:]
        if not remaining:
            continue
        anchor = slugify(heading_text)
        insert_child(item, remaining, anchor)

    if top_index is None:
        warnings.append(f"{filename}: không suy ra được số thứ tự từ tên file, cần điền tay")
        key = f"???({slug})"
    else:
        key = str(int(top_index))
        if key in items:
            warnings.append(f'{filename}: trùng key "{key}" với 1 file khác — kiểm tra lại')

    items[key] = item


## 3. Chạy xử lý

In [32]:
items = {}
warnings = []

for fn in FILES:
    process_file(SOURCE_DIR, fn, items, warnings)

if warnings:
    print("⚠️  Cảnh báo:")
    for w in warnings:
        print(" -", w)
else:
    print("Không có cảnh báo.")


Không có cảnh báo.


## 4. Kết quả — dán vào `quicklink-data.js`

Các trường `"???"` là chỗ bạn tự điền (`folder`, edition key, `label`, `path`, `index_length`).

In [33]:
output = {
    "folder": "kinhtrungbo",
    "editions": {
        "pali-vi": {
            "label": "Pali (Vi)",
            "path": "pali-vi",
            "index_length": "2",
            "items": items,
        }
    },
}

print(json.dumps(output, ensure_ascii=False, indent=2))


{
  "folder": "kinhtrungbo",
  "editions": {
    "pali-vi": {
      "label": "Pali (Vi)",
      "path": "pali-vi",
      "index_length": "2",
      "items": {
        "1": {
          "title": "MN 1. KINH PHÁP MÔN CĂN BẢN",
          "slug": "mn-001-kinh-phap-mon-can-ban"
        },
        "2": {
          "title": "MN 2. KINH TẤT CẢ CÁC LẬU HOẶC",
          "slug": "mn-002-kinh-tat-ca-cac-lau-hoac"
        },
        "3": {
          "title": "MN 3. KINH THỪA TỰ PHÁP",
          "slug": "mn-003-kinh-thua-tu-phap"
        },
        "4": {
          "title": "MN 4. KINH SỢ HÃI KHIẾP ÐẢM",
          "slug": "mn-004-kinh-so-hai-khiep-dam"
        },
        "5": {
          "title": "MN 5. KINH KHÔNG UẾ NHIỄM",
          "slug": "mn-005-kinh-khong-ue-nhiem"
        },
        "6": {
          "title": "MN 6. KINH ƯỚC NGUYỆN",
          "slug": "mn-006-kinh-uoc-nguyen"
        },
        "7": {
          "title": "MN 7. KINH VÍ DỤ TẤM VẢI",
          "slug": "mn-007-kinh-vi-du-tam-vai"
 

## 5. (Tùy chọn) Ghi ra file JSON

Chạy cell dưới nếu muốn lưu kết quả ra file thay vì chỉ copy từ output ở trên.

In [ ]:
OUT_PATH = "quicklink-data.generated.json"

with open(OUT_PATH, "w", encoding="utf-8") as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

print(f"Đã ghi: {OUT_PATH}")
